import libraries

In [1]:
import os
import json
import certifi
import numpy
import pandas as pd
import pymongo
from sqlalchemy import create_engine, text

In [2]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "#Hi10172004"
 
src_dbname = "sakila"
dst_dbname = "sakila_dw"
 
mysql_args = {
    "uid"      : user_id,
    "pwd"      : pwd,
    "hostname" : host_name,
    "dbname"   : dst_dbname
}
 
mongodb_args = {
    "user_name"        : "",
    "password"         : "",
    "cluster_name"     : "",
    "cluster_subnet"   : "",
    "cluster_location" : "local", # or could be Atlas
    "db_name"          : "sakila_catalog"
}

In [3]:
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)

In [4]:
#get data from mysql
def get_dataframe(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection)
    connection.close()
    return dframe

#set data in new db
def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
 
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
            db_connection.commit()
 
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')
            db_connection.commit()
 
    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
 
    db_connection.close()
    
def get_mongo_client(**args):
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
 
    else:
        if args["cluster_location"] == "atlas":
            connect_str  = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
 
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
 
    return client

def get_mongo_dataframe(mongo_client, db_name, collection, query):
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    return dframe

def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
 
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
 
    mongo_client.close()

In [5]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()
 
connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))
connection.commit()
connection.close()
 
print(f"Database '{dst_dbname}' created.")

Database 'sakila_dw' created.


In [6]:
#moving payments into .csv
sql_payments = "SELECT * FROM sakila.payment;"
df_payments_export = get_dataframe(user_id, pwd, host_name, src_dbname, sql_payments)
 
csv_file = os.path.join(data_dir, 'sakila_payments.csv')
df_payments_export.to_csv(csv_file, index=False)
print("Export successful")

Export successful


In [7]:
sql_films_export = """
    SELECT
        f.film_id,
        f.title,
        f.description,
        f.release_year,
        f.rental_duration,
        f.rental_rate,
        f.length            AS film_length_minutes,
        f.replacement_cost,
        f.rating,
        f.special_features,
        cat.name            AS category
    FROM sakila.film          AS f
    JOIN sakila.film_category AS fc  ON f.film_id      = fc.film_id
    JOIN sakila.category      AS cat ON fc.category_id = cat.category_id;
"""
df_films_export = get_dataframe(user_id, pwd, host_name, src_dbname, sql_films_export)
 
# Save as JSON
json_file = os.path.join(data_dir, 'sakila_films.json')
df_films_export.to_json(json_file, orient='records', indent=2)
 
# Load into MongoDB using set function
client = get_mongo_client(**mongodb_args)
 
json_files = {"films" : 'sakila_films.json'}
 
set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)
print("Film documents loaded into MongoDB.")

Exported 1,000 film documents to JSON.
Film documents loaded into MongoDB.


In [8]:
#customers
sql_customers = """
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.active,
        a.address,
        ci.city,
        co.country
    FROM sakila.customer  AS c
    JOIN sakila.address   AS a  ON c.address_id  = a.address_id
    JOIN sakila.city      AS ci ON a.city_id      = ci.city_id
    JOIN sakila.country   AS co ON ci.country_id  = co.country_id;
"""
df_customers = get_dataframe(user_id, pwd, host_name, src_dbname, sql_customers)
df_customers.head(2)

#staff
sql_staff = """
    SELECT
        s.staff_id,
        s.first_name,
        s.last_name,
        s.email,
        s.active,
        s.store_id
    FROM sakila.staff AS s;
"""
df_staff = get_dataframe(user_id, pwd, host_name, src_dbname, sql_staff)
df_staff.head(2)

#Stores
sql_stores = """
    SELECT
        st.store_id,
        ci.city         AS store_city,
        co.country      AS store_country,
        CONCAT(mgr.first_name, ' ', mgr.last_name) AS manager_name
    FROM sakila.store   AS st
    JOIN sakila.address AS a   ON st.address_id       = a.address_id
    JOIN sakila.city    AS ci  ON a.city_id            = ci.city_id
    JOIN sakila.country AS co  ON ci.country_id        = co.country_id
    JOIN sakila.staff   AS mgr ON st.manager_staff_id  = mgr.staff_id;
"""
df_stores = get_dataframe(user_id, pwd, host_name, src_dbname, sql_stores)
df_stores.head(2)

#Inventory
sql_inventory = """
    SELECT
        i.inventory_id,
        i.film_id,
        i.store_id
    FROM sakila.inventory AS i;
"""
df_inventory = get_dataframe(user_id, pwd, host_name, src_dbname, sql_inventory)
df_inventory.head(2)



,inventory_id,film_id,store_id
0,1,1,1
1,2,1,1


In [9]:
#mongodb
client = get_mongo_client(**mongodb_args)
 
query = {}
collection = "films"
 
df_films = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
df_films.head(2)
 
 
#csv
df_payments = pd.read_csv(csv_file)
df_payments['payment_date'] = pd.to_datetime(df_payments['payment_date']).dt.date
df_payments.head(2)

,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76,2.99,2005-05-25,2006-02-15 22:12:30
1,2,1,1,573,0.99,2005-05-28,2006-02-15 22:12:30


In [10]:
#rename primary key columns 
df_customers.rename(columns={"customer_id": "customer_id"}, inplace=True)
df_staff.rename(columns={"staff_id": "staff_id"}, inplace=True)
df_stores.rename(columns={"store_id": "store_id"}, inplace=True)
df_inventory.rename(columns={"inventory_id": "inventory_id"}, inplace=True)
df_films.rename(columns={"film_id": "film_id"}, inplace=True)
df_payments.rename(columns={"payment_id": "payment_id"}, inplace=True)

In [11]:
tables = [
    ('dim_customer',  df_customers, 'customer_key'),
    ('dim_film',      df_films,     'film_key'),
    ('dim_staff',     df_staff,     'staff_key'),
    ('dim_store',     df_stores,    'store_key'),
    ('dim_inventory', df_inventory, 'inventory_key'),
    ('dim_payment',   df_payments,  'payment_key'),
]
 
for table_name, dataframe, primary_key in tables:
    set_dataframe(user_id, pwd, host_name, dst_dbname, dataframe, table_name, primary_key, 'insert')

In [13]:
#select rental data
sql_fact_rentals = """
    SELECT
        r.rental_id,
        r.rental_date,
        r.return_date,
        r.customer_id,
        r.staff_id,
        r.inventory_id,
        p.payment_id,
        p.amount AS payment_amount
    FROM sakila.rental       AS r
    LEFT JOIN sakila.payment AS p ON r.rental_id = p.rental_id;
"""
df_fact_rentals = get_dataframe(user_id, pwd, host_name, src_dbname, sql_fact_rentals)
df_fact_rentals.head(2)

#determine rental days
df_fact_rentals['rental_date'] = pd.to_datetime(df_fact_rentals['rental_date'])
df_fact_rentals['return_date'] = pd.to_datetime(df_fact_rentals['return_date'])
df_fact_rentals['rental_days'] = (
    (df_fact_rentals['return_date'] - df_fact_rentals['rental_date'])
    .dt.total_seconds() / 86400
).round(1)

#getting primary keys from dim_Date
sql_dim_date = "SELECT date_key, full_date FROM sakila_dw.dim_date;"
df_dim_date = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_date)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

# lookup rental_date_key
df_dim_rental_date = df_dim_date.rename(columns={"date_key": "rental_date_key", "full_date": "rental_date"})
df_fact_rentals.rental_date = df_fact_rentals.rental_date.astype('datetime64[ns]').dt.date
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_rental_date, on='rental_date', how='left')
df_fact_rentals.drop(['rental_date'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
# lookup return_date_key
df_dim_return_date = df_dim_date.rename(columns={"date_key": "return_date_key", "full_date": "return_date"})
df_fact_rentals.return_date = df_fact_rentals.return_date.astype('datetime64[ns]').dt.date
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_return_date, on='return_date', how='left')
df_fact_rentals.drop(['return_date'], axis=1, inplace=True)
df_fact_rentals.head(2)

#look up primary keys from other dimension tables
sql_dim_customers = "SELECT customer_key, customer_id FROM sakila_dw.dim_customer;"
df_dim_customers = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_customers)
df_dim_customers.head(2)
 
sql_dim_staff = "SELECT staff_key, staff_id FROM sakila_dw.dim_staff;"
df_dim_staff = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_staff)
df_dim_staff.head(2)
 
sql_dim_inventory = "SELECT inventory_key, inventory_id, film_id FROM sakila_dw.dim_inventory;"
df_dim_inventory = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_inventory)
df_dim_inventory.head(2)
 
sql_dim_films = "SELECT film_key, film_id FROM sakila_dw.dim_film;"
df_dim_films = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_films)
df_dim_films.head(2)
 
sql_dim_payments = "SELECT payment_key, payment_id FROM sakila_dw.dim_payment;"
df_dim_payments = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_payments)
df_dim_payments.head(2)
 
sql_dim_staff_store = """
    SELECT ds.staff_key, dst.store_key
    FROM sakila_dw.dim_staff AS ds
    JOIN sakila_dw.dim_store AS dst ON ds.store_id = dst.store_id;
"""
df_dim_staff_store = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_staff_store)
df_dim_staff_store.head(2)

##look up the corresponding surrogate keys and merge

# Merge customer_key
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_customers, on='customer_id', how='left')
df_fact_rentals.drop(['customer_id'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
# Merge staff_key
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_staff, on='staff_id', how='left')
df_fact_rentals.drop(['staff_id'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
# Merge inventory_key (also gives us film_id for the next lookup)
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_inventory, on='inventory_id', how='left')
df_fact_rentals.drop(['inventory_id'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
# Merge film_key via film_id (rental → inventory → film)
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_films, on='film_id', how='left')
df_fact_rentals.drop(['film_id'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
# Merge store_key via staff_key
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_staff_store, on='staff_key', how='left')
df_fact_rentals.head(2)
 
# Merge payment_key
df_fact_rentals = pd.merge(df_fact_rentals, df_dim_payments, on='payment_id', how='left')
df_fact_rentals.drop(['payment_id'], axis=1, inplace=True)
df_fact_rentals.head(2)
 
 

DatabaseError: Execution failed on sql 'SELECT date_key, full_date FROM sakila_dw.dim_date;': (pymysql.err.ProgrammingError) (1146, "Table 'sakila_dw.dim_date' doesn't exist")
[SQL: SELECT date_key, full_date FROM sakila_dw.dim_date;]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [14]:
table_name = "fact_rentals"
primary_key = "rental_key"
db_operation = "insert"
 
set_dataframe(user_id, pwd, host_name, dst_dbname, df_fact_rentals, table_name, primary_key, db_operation)


In [15]:
sql_test = """
    SELECT
        CONCAT(dc.first_name, ' ', dc.last_name) AS `Customer Name`,
        dc.city                                   AS `City`,
        dc.country                                AS `Country`,
        COUNT(fr.rental_key)                      AS `Total Rentals`,
        SUM(fr.payment_amount)                    AS `Total Spent`,
        AVG(fr.rental_days)                       AS `Avg Rental Days`
    FROM `{0}`.fact_rentals  AS fr
    INNER JOIN `{0}`.dim_customer AS dc ON fr.customer_key = dc.customer_key
    GROUP BY dc.customer_key, dc.first_name, dc.last_name, dc.city, dc.country
    ORDER BY `Total Spent` DESC
    LIMIT 10;
"""
 
df_test = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_test)
df_test.head()

DatabaseError: Execution failed on sql '
    SELECT
        CONCAT(dc.first_name, ' ', dc.last_name) AS `Customer Name`,
        dc.city                                   AS `City`,
        dc.country                                AS `Country`,
        COUNT(fr.rental_key)                      AS `Total Rentals`,
        SUM(fr.payment_amount)                    AS `Total Spent`,
        AVG(fr.rental_days)                       AS `Avg Rental Days`
    FROM `sakila_dw`.fact_rentals  AS fr
    INNER JOIN `sakila_dw`.dim_customer AS dc ON fr.customer_key = dc.customer_key
    GROUP BY dc.customer_key, dc.first_name, dc.last_name, dc.city, dc.country
    ORDER BY `Total Spent` DESC
    LIMIT 10;
': (pymysql.err.OperationalError) (1054, "Unknown column 'fr.customer_key' in 'on clause'")
[SQL: 
    SELECT
        CONCAT(dc.first_name, ' ', dc.last_name) AS `Customer Name`,
        dc.city                                   AS `City`,
        dc.country                                AS `Co